# Expected Wait

### Definition
The proportion of the time during the Day Part in which an arriving passenger would experience an Actual Headway that is greater than the Scheduled Headway.

### Reporting Grain
Service Pattern, stop, Day Part, and Reporting Period

### Notes
Also including expected wait time matching Chicago Transit Authority performance metrics.

In [ ]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import shapely.wkt

from src.data_loading import (
    get_matching_shapes_feeds,
    get_schedule_rt_headways,
    get_schedule_feed_entries,
)
from src.metric_calculation import calculate_expected_wait
from src.secrets import carto_map_parameters

### Parameters

In [ ]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

In [ ]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

### Implementation

In [ ]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))


In [ ]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

In [ ]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

In [ ]:
# Get the realtime headway between every pair of consecutive trips, at every
# stop on the pattern rather than only the timepoints
schedule_rt_headways = get_schedule_rt_headways(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
schedule_rt_headways


In [ ]:
schedule_rt_headways.loc[(schedule_rt_headways.stop_sequence == 2) & (schedule_rt_headways.first_actual_departure.dt.hour == 14)].sort_values(by="first_actual_departure")

In [ ]:
# The query carries the headway itself, so the distribution is ready to look at
schedule_rt_headways["rt_headway_seconds"].describe()

### Results

In [ ]:
# Expected and excess wait per stop and day part, pooled over every selected date.
#
# Each headway is assigned to a day part by the clock hour it opens on, so a
# trip the schedule publishes as 24:15 counts in Owl the following morning.
# Day parts a stop saw no service in are absent rather than present and empty.
headway_summary = calculate_expected_wait(schedule_rt_headways)
headway_summary

In [ ]:
headway_summary.loc[headway_summary.time_period == "Midday"].describe()

In [ ]:
# Total headways observed per day
# note fewer headways on start and end of the week
schedule_rt_headways["first_actual_departure"].dt.date.value_counts()

In [ ]:
# The longest gaps in service, which are what riders actually notice
(
    schedule_rt_headways
    .sort_values("rt_headway_seconds", ascending=False)
    .head(20)
    [["shape_id", "stop_sequence", "stop_id", "first_actual_departure", "second_actual_departure", "rt_headway_seconds"]]
)

In [ ]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by how much of the day part would have left a rider waiting
# longer than the schedule promised.
MAP_TIME_PERIOD = "Midday"

# The headways query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_headways[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    headway_summary.loc[headway_summary["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# percent_time_excess_wait is a proportion; the map reads better in percent. The
# colour and the tooltip use the one column, so the legend matches what is hovered
stops_for_period["percent_time_excess_wait"] = (
    stops_for_period["percent_time_excess_wait"] * 100
).round(1)
for column in [
    "scheduled_expected_wait_seconds",
    "rt_expected_wait_seconds",
    "excess_wait_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
period_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    **carto_map_parameters(),
    name="service patterns",
)
stops_for_period.explore(
    m=period_map,
    column="percent_time_excess_wait",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "scheduled_expected_wait_seconds",
        "rt_expected_wait_seconds",
        "excess_wait_seconds",
        "percent_time_excess_wait",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Scheduled expected wait (s)",
            "Actual expected wait (s)",
            "Excess wait (s)",
            "% of time waiting over schedule",
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} spent waiting longer than scheduled"
    },
    name=f"{MAP_TIME_PERIOD} excess wait",
)
period_map.save(f"expected_wait_{MAP_TIME_PERIOD.lower()}.html")
period_map